In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

try:
    response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",  
        messages=[{"role": "user", "content": "Hello! Confirming that the client works."}]
    )
    print("Success! Response from Gemini:")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"Error: {e}")
    print("Double check your API key and ensure your .env file is in the right directory.")

Success! Response from Gemini:
Hello!

To confirm, could you please specify:

1.  **Which client** you are referring to? (e.g., a specific software application, a service, a project, a customer account, etc.)
2.  **What aspect of "working"** you'd like me to check? (e.g., Is it functional, responsive, integrated, processing data, etc.)

Once I have those details, I can provide a more accurate confirmation!


In [8]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [9]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That's great you've discovered it and are interested!

Whether you can join now really depends on a few things about the specific course. To give you the most accurate answer, could you tell me:

1.  **What is the name of the course?**
2.  **Where did you discover it (e.g., a specific university website, Coursera, edX, Udemy, an internal company platform, etc.)?**

In the meantime, here are some general scenarios:

*   **Self-Paced Courses:** If it's a self-paced course, you can usually enroll and start anytime.
*   **Cohort-Based Courses (with specific start dates):**
    *   **If it hasn't started yet:** You can likely still enroll, but check the registration deadline.
    *   **If it has already started:** You might be able to join late, but you'll have missed some material and assignments. Some courses don't allow late enrollment at all. You might have to wait for the next offering.
*   **Courses with Rolling Enrollment:** Some programs have rolling enrollment, meaning you can join

In [10]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [11]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [12]:
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [13]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [14]:
import requests
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [15]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1346

In [16]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [17]:
search_results=index.search(question,
             filter_dict={'course':'llm-zoomcamp'},
             num_results=5
             )

In [18]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [19]:
search_results = search(question)

In [20]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [21]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [23]:
def build_prompt(question,search_results):
    context=build_context(search_results)
    prompt=USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [24]:
prompt=build_prompt(question,search_results)

In [25]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [29]:
response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )

In [30]:
response.choices[0].message.content

'Yes, you can join and start learning right away!\n\nThe course materials (videos, notebooks, and code) are available, and you can start whenever you want. You don\'t need a confirmation email or formal registration to begin learning and submitting homework.\n\nHowever, if you want to **receive a certificate**, there are specific conditions:\n*   You must submit your projects and homework **while submissions are still being accepted** (before the deadlines listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/)).\n*   You need to participate with a "live" cohort because certificates require you to peer-review other students\' capstone projects, which is only possible when the course is actively running and the peer-review list is compiled.\n\nSelf-paced completion without meeting these deadlines and peer-review requirements will not result in a certificate.\n\nTo get started, you can refer to the [LLM Zoomcamp docs](https://datatalks.club/docs/cour

In [33]:
response.usage

CompletionUsage(completion_tokens=277, prompt_tokens=514, total_tokens=1836, completion_tokens_details=None, prompt_tokens_details=None)

In [35]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price +
    response.usage.completion_tokens * output_price
)

cost

0.001632

In [43]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=message_history
)

In [44]:
response.choices[0].message.content

'Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available.'

In [49]:
def llm(instructions, prompt,model):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

In [50]:
def rag(question,model="gemini-2.5-flash"):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    return llm(INSTRUCTIONS, prompt, model=model)

In [51]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. The videos and GitHub materials are available, and you can start whenever you want.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can only get a certificate if you finish the course with a "live" cohort, as you need to peer-review projects at the time the course is running.
